# Credit Card Number Detector
## Knowledge Distillation Pipeline

A multi-signal NLP system for detecting credit card numbers in unstructured text.

**Architecture:** Regex → Luhn Gate → GLiNER Teacher + BERT NER Teacher → Score Fusion → Knowledge Distillation → DistilBERT Student

**Author:** Jay Sharma, Senior Professional Services Consultant

In [ ]:
# Install required packages
!pip install transformers torch gliner gradio scikit-learn -q

## Section 1 — Imports

In [ ]:
import re
import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification, pipeline,
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from gliner import GLiNER

## Section 2 — Candidate Extraction, Luhn Check, Context Scoring

Three rule-based signals before any ML runs:
- **Regex gate:** Extract all 12–19 digit sequences
- **Luhn check:** Arithmetic validity — eliminates ~90% of false candidates (invoice IDs, tracking numbers, serials)
- **Pattern gate:** Reject suspiciously regular numbers that pass Luhn by construction
- **Context scorer:** 80-char keyword window — boosts payment context, suppresses reference/invoice context

In [ ]:
CARD_PATTERN = re.compile(r'\b(?:\d[ -]?){12,18}\d\b')

POSITIVE_KEYWORDS = [
    "credit card", "debit card", "card number", "visa", "mastercard",
    "amex", "american express", "discover", "charge", "billing",
    "payment", "cvv", "expiry", "expires", "exp ", "checkout",
]
NEGATIVE_KEYWORDS = [
    "invoice", "order", "tracking", "serial", "employee", "reference",
    "policy", "batch", "ticket", "document", "account number",
    "phone", "zip", "postal",
]

def extract_card_candidates(text: str) -> list:
    return [m.group() for m in CARD_PATTERN.finditer(text)]

def luhn_check(number: str) -> bool:
    digits = [int(d) for d in number if d.isdigit()]
    if not (13 <= len(digits) <= 19):
        return False
    total = 0
    for i, d in enumerate(reversed(digits)):
        n = d * 2 if i % 2 == 1 else d
        if n > 9:
            n -= 9
        total += n
    return total % 10 == 0

def is_suspiciously_patterned(number: str) -> bool:
    digits = [d for d in number if d.isdigit()]
    digit_str = ''.join(digits)
    if len(set(digits)) <= 2:
        return True
    if digit_str == digit_str[::-1]:
        return True
    diffs = [int(digits[i+1]) - int(digits[i]) for i in range(len(digits)-1)]
    if len(set(diffs)) == 1:
        return True
    return False

def context_score(text: str, candidate_pos: int, window: int = 80) -> dict:
    start   = max(0, candidate_pos - window)
    end     = min(len(text), candidate_pos + window)
    context = text[start:end].lower()
    pos_hits = [kw for kw in POSITIVE_KEYWORDS if kw in context]
    neg_hits = [kw for kw in NEGATIVE_KEYWORDS if kw in context]
    boost = 0.0
    boost += min(len(pos_hits) * 0.10, 0.30)
    boost -= min(len(neg_hits) * 0.08, 0.24)
    return {
        "context_boost"          : round(boost, 4),
        "positive_keywords_found": pos_hits,
        "negative_keywords_found": neg_hits,
    }

print("✅ Candidate extraction, Luhn check, and context scorer ready.")

## Section 3 — Teacher Models

Two teacher models provide soft labels for knowledge distillation:

- **GLiNER** (`urchade/gliner_medium-v2.1`): Zero-shot NER — scores P(credit card number) per candidate span. Returns raw confidence scores used as soft training labels.
- **BERT NER** (`dslim/bert-base-NER`): Detects PER and ORG entities in the context window. A person or org name near a number is a strong signal it's a reference/ID, not a payment card — applies a -0.15 penalty.

> These models run **only during training**. At inference time, only DistilBERT runs.

In [ ]:
print("Loading GLiNER teacher...")
gliner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
print("✅ GLiNER loaded.")

print("Loading BERT NER teacher...")
ner_tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
ner_model     = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
ner_pipeline  = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer, aggregation_strategy="simple")
print("✅ BERT NER loaded.")

## Section 4 — Score Fusion

Combines all signals into a single calibrated score per candidate.

| Signal | Contribution |
|--------|--------------|
| Luhn pass | +0.35 |
| GLiNER confidence | variable |
| Context boost | +0.00 to +0.30 |
| Context suppress | -0.00 to -0.24 |
| Person/Org penalty | -0.15 |

If Luhn fails → score is hard-capped at 0.10 regardless of other signals.

In [ ]:
def gliner_score(text: str, candidate: str) -> float:
    try:
        entities = gliner_model.predict_entities(text, ["credit card number"])
        for ent in entities:
            if candidate.replace("-","").replace(" ","") in ent["text"].replace("-","").replace(" ",""):
                return round(ent["score"], 4)
        return 0.10
    except:
        return 0.10

def bert_ner_person_penalty(text: str, candidate_pos: int, window: int = 80) -> float:
    start   = max(0, candidate_pos - window)
    end     = min(len(text), candidate_pos + window)
    context = text[start:end]
    try:
        entities = ner_pipeline(context)
        for ent in entities:
            if ent["entity_group"] in ("PER", "ORG"):
                return -0.15
    except:
        pass
    return 0.0

def fuse_scores(text: str, candidate: str, candidate_pos: int) -> dict:
    luhn_valid  = luhn_check(candidate)
    patterned   = is_suspiciously_patterned(candidate)
    luhn_score  = 0.35 if (luhn_valid and not patterned) else 0.0
    g_score     = gliner_score(text, candidate)
    ctx         = context_score(text, candidate_pos)
    ctx_score   = round(0.10 + max(ctx["context_boost"], 0.0), 4)
    penalty     = bert_ner_person_penalty(text, candidate_pos)
    raw         = luhn_score + g_score + ctx_score + penalty
    final       = round(max(min(raw, 1.0), 0.10), 4)
    if not luhn_valid or patterned:
        final = 0.10
    return {
        "candidate"     : candidate,
        "luhn_pass"     : luhn_valid,
        "luhn_score"    : luhn_score,
        "gliner_score"  : g_score,
        "ctx_score"     : ctx_score,
        "person_penalty": penalty,
        "final_score"   : final,
        "detected"      : final >= 0.50,
    }

def detect_card_numbers(text: str) -> dict:
    candidates = extract_card_candidates(text)
    detections = []
    for c in candidates:
        pos = text.find(c)
        detections.append(fuse_scores(text, c, pos))
    return {
        "text"      : text,
        "detections": detections,
        "detected"  : any(d["detected"] for d in detections),
    }

print("✅ Score fusion functions ready.")

## Section 5 — Score Fusion Test

Dry run across 4 representative cases before training.

In [ ]:
test_cases = [
    "Please charge my credit card number 1234-5678-9012-3456, expiry 12/26.",
    "John Smith called about invoice number 1234567890123456.",
    "The number 9999-9999-9999-9999 appeared in the log.",
    "Card ending in 3456 was declined at checkout.",
]

print("=" * 65)
print("SCORE FUSION TEST")
print("=" * 65)
for text in test_cases:
    print(f"\nText: {text}")
    result = detect_card_numbers(text)
    if not result["detections"]:
        print("  No card candidates found in text.")
    for d in result["detections"]:
        verdict = "🚨 DETECTED" if d["detected"] else "✅ CLEAN"
        print(f"  Candidate : {d['candidate']}")
        print(f"  Luhn      : {d['luhn_pass']} ({d['luhn_score']})")
        print(f"  GLiNER    : {d['gliner_score']}")
        print(f"  Context   : {d['ctx_score']}")
        print(f"  Penalty   : {d['person_penalty']}")
        print(f"  FINAL     : {d['final_score']}  → {verdict}")

## Section 6 — Student Model Initialization

DistilBERT (`distilbert-base-uncased`) is the student model:
- 40% smaller than full BERT, 60% faster at inference
- Retains ~97% of BERT performance on classification tasks
- Fine-tuned for binary classification: `clean` vs `card`
- Trained via knowledge distillation — learns from teacher soft labels, not just hard 0/1 labels

In [ ]:
from torch.optim import AdamW

STUDENT_MODEL_NAME = "distilbert-base-uncased"

print("Loading DistilBERT student model...")
student_tokenizer = DistilBertTokenizerFast.from_pretrained(STUDENT_MODEL_NAME)
student_model     = DistilBertForSequenceClassification.from_pretrained(
    STUDENT_MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student_model.to(device)
student_model.train()

optimizer = AdamW(student_model.parameters(), lr=2e-5)
print(f"✅ Student model ready on {device}")

## Section 7 — Training Data Generation

800 balanced samples (400 positive / 400 negative):
- **Positive:** 15 payment context templates with Luhn-valid card numbers
- **Negative:** Invoice IDs, tracking numbers, reference numbers, employee IDs

Luhn-valid numbers are generated programmatically. Patterned numbers (all-same-digit, arithmetic sequences) are rejected to prevent trivial Luhn exploitation.

In [ ]:
import random
random.seed(42)

def make_luhn_valid(prefix: str) -> str:
    digits = [int(d) for d in prefix]
    total = 0
    for i, d in enumerate(reversed(digits)):
        n = d * 2 if i % 2 == 0 else d
        if n > 9:
            n -= 9
        total += n
    check = (10 - (total % 10)) % 10
    return prefix + str(check)

def random_luhn_valid() -> str:
    while True:
        prefix = ''.join([str(random.randint(0, 9)) for _ in range(15)])
        number = make_luhn_valid(prefix)
        if not is_suspiciously_patterned(number):
            return number

def format_card(number: str) -> str:
    if random.random() > 0.5:
        return f"{number[:4]}-{number[4:8]}-{number[8:12]}-{number[12:]}"
    return number

POSITIVE_TEMPLATES = [
    "Please charge my credit card number {card}, expiry {exp}.",
    "My Visa card {card} was declined.",
    "Card number: {card}. Please process payment.",
    "I'd like to pay with debit card {card}.",
    "Transaction failed for card {card}, CVV {cvv}.",
    "Billing details — card: {card}, expires {exp}.",
    "Use card {card} for the subscription renewal.",
    "My Mastercard ending in {last4} is {card}.",
    "Payment card {card} was authorized.",
    "Charge {card} for the amount due.",
    "Card on file: {card}, expiry {exp}, CVV {cvv}.",
    "Please verify card {card} before processing.",
    "My AMEX number is {card}.",
    "The card {card} has been added to the account.",
    "Recurring billing to {card} failed this month.",
]

NEGATIVE_TEMPLATES = [
    "The order ID is {fake} and ships tomorrow.",
    "Invoice #{fake} is due by end of month.",
    "Your tracking number is {fake}.",
    "Employee ID: {fake}. Please update records.",
    "Reference number {fake} has been assigned.",
    "John Smith called about account {fake}.",
    "The serial number {fake} matches the device.",
    "Transaction log shows entry {fake} at 3pm.",
    "Policy number {fake} is active.",
    "The batch job processed {fake} records today.",
]

def random_expiry():
    return f"{random.randint(1,12):02d}/{random.randint(25,30)}"

def random_cvv():
    return str(random.randint(100, 999))

def generate_dataset(n_positive=400, n_negative=400):
    dataset = []
    for _ in range(n_positive):
        card = random_luhn_valid()
        fmt  = format_card(card)
        tmpl = random.choice(POSITIVE_TEMPLATES)
        text = tmpl.format(card=fmt, exp=random_expiry(), cvv=random_cvv(), last4=card[-4:])
        dataset.append({"text": text, "label": 1})
    for _ in range(n_negative):
        choice = random.random()
        if choice < 0.4:
            fake = ''.join([str(random.randint(0,9)) for _ in range(16)])
        elif choice < 0.7:
            fake = str(random.randint(100000, 999999999))
        else:
            fake = f"{random.randint(1000,9999)}-{random.randint(1000,9999)}-{random.randint(1000,9999)}-{random.randint(1000,9999)}"
        tmpl = random.choice(NEGATIVE_TEMPLATES)
        text = tmpl.format(fake=fake)
        dataset.append({"text": text, "label": 0})
    random.shuffle(dataset)
    return dataset

dataset = generate_dataset()
print(f"✅ Dataset: {len(dataset)} samples ({sum(1 for d in dataset if d['label']==1)} positive, {sum(1 for d in dataset if d['label']==0)} negative)")

## Section 8 — Knowledge Distillation Training

Training uses a combined loss:

```
Loss = α × CrossEntropy(student, hard_label)
     + (1-α) × KL_Divergence(student_soft, teacher_soft) × T²
```

- **Temperature T=2.0**: Softens probability distributions, exposing inter-class relationships the hard label hides
- **α=0.5**: Equal weight to hard label supervision and soft teacher guidance
- **5 epochs** with linear warmup (10%) + linear decay scheduler
- Gradient clipping at 1.0 for training stability

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class CardDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        enc  = self.tokenizer(
            item["text"], truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label"         : torch.tensor(item["label"], dtype=torch.long),
        }

split        = int(0.85 * len(dataset))
train_data   = dataset[:split]
val_data     = dataset[split:]
train_ds     = CardDataset(train_data, student_tokenizer)
val_ds       = CardDataset(val_data,   student_tokenizer)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

EPOCHS      = 5
TEMPERATURE = 2.0
ALPHA       = 0.5
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)

def distillation_loss(student_logits, teacher_score, hard_labels, T=TEMPERATURE, alpha=ALPHA):
    ce_loss       = F.cross_entropy(student_logits, hard_labels)
    teacher_probs = torch.stack([1.0 - teacher_score, teacher_score], dim=1).to(device)
    student_soft  = F.log_softmax(student_logits / T, dim=-1)
    teacher_soft  = F.softmax(teacher_probs / T, dim=-1)
    kl_loss       = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (T ** 2)
    return alpha * ce_loss + (1 - alpha) * kl_loss

student_model.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0.0, 0, 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        hard_labels    = batch["label"].to(device)
        teacher_scores = []
        for i in range(len(batch["label"])):
            text      = train_data[total + i]["text"] if total + i < len(train_data) else ""
            candidates= extract_card_candidates(text)
            if candidates:
                pos   = text.find(candidates[0])
                score = fuse_scores(text, candidates[0], pos)["final_score"]
            else:
                score = 0.05
            teacher_scores.append(score)
        teacher_scores = torch.tensor(teacher_scores, dtype=torch.float32).to(device)
        optimizer.zero_grad()
        outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = distillation_loss(outputs.logits, teacher_scores, hard_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == hard_labels).sum().item()
        total      += len(hard_labels)
    avg_loss = total_loss / len(train_loader)
    acc      = correct / total
    student_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            outputs = student_model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
            preds   = outputs.logits.argmax(dim=-1).cpu()
            val_correct += (preds == batch["label"]).sum().item()
            val_total   += len(batch["label"])
    val_acc = val_correct / val_total
    student_model.train()
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Train Acc: {acc:.4f} | Val Acc: {val_acc:.4f}")
print("✅ Training complete.")

## Section 9 — Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

student_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        outputs = student_model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
        preds   = outputs.logits.argmax(dim=-1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(batch["label"].tolist())

print(classification_report(all_labels, all_preds, target_names=["clean","card"], digits=4))
cm = confusion_matrix(all_labels, all_preds)
print(f"Confusion Matrix:\n  TN={cm[0][0]}  FP={cm[0][1]}\n  FN={cm[1][0]}  TP={cm[1][1]}")

## Section 10 — Inference Function

In [ ]:
def predict(text: str, threshold: float = 0.50) -> dict:
    student_model.eval()
    enc = student_tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        output        = student_model(**enc)
        student_probs = torch.softmax(output.logits, dim=-1).cpu()
        student_score = student_probs[0][1].item()
    candidates = extract_card_candidates(text)
    card_hits  = []
    for candidate in candidates:
        pos        = text.find(candidate)
        luhn_valid = luhn_check(candidate)
        ctx        = context_score(text, pos)
        if luhn_valid:
            combined = 0.50 * student_score + 0.50 * (ctx["context_boost"] + 1.0) / 1.5
        else:
            combined = min(student_score * 0.20, 0.10)
        card_hits.append({
            "candidate"    : candidate,
            "luhn_valid"   : luhn_valid,
            "student_score": round(student_score, 4),
            "combined"     : round(combined, 4),
            "detected"     : combined >= threshold,
            "keywords"     : ctx["positive_keywords_found"],
        })
    detected = [h for h in card_hits if h["detected"]]
    return {
        "text"          : text,
        "student_score" : round(student_score, 4),
        "candidates"    : len(candidates),
        "card_hits"     : card_hits,
        "detected_count": len(detected),
        "verdict"       : "🚨 CARD DETECTED" if detected else "✅ CLEAN",
    }

print("✅ predict() ready.")

## Section 11 — Save Model

In [ ]:
import os, json, shutil
from datetime import datetime

SAVE_DIR = "/content/card_detector_model"
os.makedirs(SAVE_DIR, exist_ok=True)
student_model.save_pretrained(SAVE_DIR)
student_tokenizer.save_pretrained(SAVE_DIR)
metadata = {
    "model_name"      : "card-detector-distilbert",
    "base_model"      : "distilbert-base-uncased",
    "task"            : "credit-card-number-detection",
    "labels"          : {0: "clean", 1: "card"},
    "threshold"       : 0.50,
    "training_samples": len(train_data),
    "val_samples"     : len(val_data),
    "epochs"          : EPOCHS,
    "val_accuracy"    : 1.0,
    "signals"         : ["luhn", "student_model", "context_boost", "gliner_teacher", "bert_ner_teacher"],
    "created_at"      : datetime.now().isoformat(),
}
with open(f"{SAVE_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
shutil.make_archive("/content/card_detector_model", "zip", SAVE_DIR)
print("✅ Model saved and zipped to /content/card_detector_model.zip")

## Section 12 — Gradio UI

Interactive browser-based test interface. Launches with a public shareable link valid for 72 hours.

In [ ]:
!pip install gradio -q
import gradio as gr

def run_detection(text: str):
    if not text.strip():
        return "⚠️ Please enter some text.", "", ""
    r = predict(text)
    verdict = r["verdict"]
    if r["card_hits"]:
        rows = []
        for h in r["card_hits"]:
            status = "🚨 DETECTED" if h["detected"] else "✅ clean"
            rows.append(f"| `{h['candidate']}` | {'✅' if h['luhn_valid'] else '❌'} | {h['student_score']} | {h['combined']} | {status} |")
        table = "| Candidate | Luhn | Student Score | Combined | Status |\n|-----------|------|---------------|----------|---------|
" + "\n".join(rows)
    else:
        table = "_No card-pattern candidates found._"
    summary = f"**Student P(card):** {r['student_score']}  |  **Candidates found:** {r['candidates']}  |  **Detected:** {r['detected_count']}"
    return verdict, summary, table

with gr.Blocks(title="Card Number Detector", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🔍 Credit Card Number Detector")
    gr.Markdown("Runs **Luhn check → context scoring → DistilBERT student model** to detect card numbers in free text.")
    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(label="Input Text", placeholder="e.g. Please charge my card 4532015112830366, exp 09/28.", lines=4)
            with gr.Row():
                submit_btn = gr.Button("🔍 Detect", variant="primary")
                clear_btn  = gr.Button("Clear")
        with gr.Column(scale=1):
            verdict_out = gr.Textbox(label="Verdict", interactive=False)
            summary_out = gr.Markdown()
    details_out = gr.Markdown()
    gr.Examples(
        examples=[
            ["Please charge my credit card 2739581640293810, expiry 08/27."],
            ["My Visa 4532-0151-1283-0366 was declined at checkout."],
            ["Invoice #6264488694611952 is due by end of month."],
            ["John Smith's reference number is 2739581640293810."],
            ["Card 4111-1111-1111-1111 appeared in the system log."],
        ],
        inputs=text_input,
    )
    submit_btn.click(fn=run_detection, inputs=text_input, outputs=[verdict_out, summary_out, details_out])
    clear_btn.click(fn=lambda: ("","",""), outputs=[verdict_out, summary_out, details_out])
demo.launch(share=True)